# IDs de Propiedades Repetidos

## 1. Comprensión de la Estructura de los Datos
Antes de analizar los "duplicados", debemos entender qué representa cada columna:

| Columna | Significado | Tipo |
|--------|---------|------|
| `id` | **Identificador de propiedad** — identifica el edificio/terreno físico | Identificador (no es una característica) |
| `date` | **Fecha de venta** — cuándo ocurrió la transacción | Temporal |
| `price` | **Precio de venta** — el monto de la transacción | Variable objetivo |
| Otras columnas | Características de la propiedad | Características |

**Distinción crítica**: El `id` identifica **propiedades**, no **ventas**. Una propiedad puede venderse múltiples veces, generando múltiples registros con el mismo `id` pero con valores diferentes de `date` y `price`.

## 2. Carga de Datos

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as plt

In [ ]:
import kagglehubfrom pathlib import Pathpath = kagglehub.dataset_download("harlfoxem/housesalesprediction")csv_path = Path(path) / "kc_house_data.csv"df = pd.read_csv(csv_path)print(f"Dataset shape: {df.shape}")

Dataset shape: (21613, 21)


## 3. Identificación de IDs Repetidos

In [ ]:
id_counts = df["id"].value_counts()repeated_ids = id_counts[id_counts > 1]print(f"Total records: {len(df):,}")print(f"Unique property IDs: {df['id'].nunique():,}")print(f"Properties with multiple sales: {len(repeated_ids):,}")print(f"Records from repeated IDs: {df['id'].isin(repeated_ids.index).sum():,}")

Total records: 21,613
Unique property IDs: 21,436
Properties with multiple sales: 176
Records from repeated IDs: 353


In [ ]:
# As percentagesrepeated_records = df[df["id"].isin(repeated_ids.index)]pct_repeated_records = 100 * len(repeated_records) / len(df)pct_repeated_ids = 100 * len(repeated_ids) / df["id"].nunique()print(f"\nPercentage of records with repeated IDs: {pct_repeated_records:.2f}%")print(f"Percentage of properties sold more than once: {pct_repeated_ids:.2f}%")


Percentage of records with repeated IDs: 1.63%
Percentage of properties sold more than once: 0.82%


In [ ]:
# How many times are IDs repeated?id_counts.value_counts().sort_index()

count
1    21260
2      175
3        1
Name: count, dtype: int64

## 4. ¿Qué Representan Estos Registros Repetidos?
Examinemos ejemplos específicos para entender qué difiere entre registros con el mismo ID.

In [ ]:
example_ids = repeated_ids.head(3).index.tolist()for prop_id in example_ids:    print(f"\n{'='*70}")    print(f"Property ID: {prop_id}")    print("="*70)    records = df[df["id"] == prop_id].sort_values("date")    display(records[["id", "date", "price", "sqft_living", "bedrooms", "bathrooms", "grade"]])


Property ID: 795000620


,id,date,price,sqft_living,bedrooms,bathrooms,grade
17602,795000620,20140924T000000,115000.0,1080,3,1.0,5
17603,795000620,20141215T000000,124000.0,1080,3,1.0,5
17604,795000620,20150311T000000,157000.0,1080,3,1.0,5



Property ID: 3323059027


,id,date,price,sqft_living,bedrooms,bathrooms,grade
6630,3323059027,20140528T000000,326000.0,1720,3,2.75,7
6631,3323059027,20150225T000000,340000.0,1720,3,2.75,7



Property ID: 1450100390


,id,date,price,sqft_living,bedrooms,bathrooms,grade
10272,1450100390,20140905T000000,125000.0,920,3,1.0,6
10273,1450100390,20150316T000000,208000.0,920,3,1.0,6


### 4.1 Análisis: ¿Qué Cambia Entre Registros?
Para propiedades vendidas múltiples veces, esperamos:
- **`date`**: Siempre diferente (distintas fechas de transacción)
- **`price`**: Usualmente diferente (condiciones del mercado, negociación)
- **Características físicas**: En su mayoría sin cambios (el mismo edificio)

Verifiquemos esto sistemáticamente:

In [ ]:
# Analyze what columns change between sales of the same propertyfeature_cols = [col for col in df.columns if col not in ["id", "date", "price"]]properties_with_changes = 0changed_features_count = {col: 0 for col in feature_cols}for prop_id in repeated_ids.index:    records = df[df["id"] == prop_id]        has_any_change = False    for col in feature_cols:        if records[col].nunique() > 1:            changed_features_count[col] += 1            has_any_change = True        if has_any_change:        properties_with_changes += 1print(f"Properties where features changed between sales: {properties_with_changes} / {len(repeated_ids)}")print(f"\nFeatures that changed (and how often):")for col, count in sorted(changed_features_count.items(), key=lambda x: -x[1]):    if count > 0:        print(f"  {col}: {count} properties")

Properties where features changed between sales: 0 / 176

Features that changed (and how often):


### 4.2 Conclusión: Son Transacciones de Venta Distintas
El análisis confirma:
1. **Misma propiedad, diferentes ventas**: Cada registro representa una transacción inmobiliaria legítima
2. Las características físicas permanecen en su mayoría constantes. Los cambios en características entre ventas representan renovaciones o actualizaciones, que son señales válidas para el modelo.
3. **El precio y la fecha difieren**: Estos son los valores específicos de cada transacción

**Estos NO son duplicados en el sentido de calidad de datos** — son observaciones válidas de diferentes eventos de venta.

## 5. La Columna `id`: Por Qué Debe Eliminarse
Independientemente de los IDs repetidos, la columna `id` **debe eliminarse** como característica. He aquí por qué:

### 5.1 Razón Conceptual: Los Identificadores No Son Características
El `id` es un **identificador de propiedad** — una etiqueta arbitraria asignada a cada edificio. No contiene información intrínseca sobre el valor de la propiedad. Usarlo como característica sería como usar el número de seguridad social de una persona para predecir su salario.

### 5.2 Razón Técnica: La Codificación Es Ineficaz y Propensa al Sobreajuste
Con ~21,000 IDs únicos:
- **Codificación one-hot**: Crearía ~21,000 características binarias — computacionalmente prohibitivo y lleva a sobreajuste extremo
- **Codificación de etiquetas**: Impondría una relación ordinal artificial (la propiedad 1000 no es "mayor que" la propiedad 500)
- **Codificación por objetivo**: Causaría fuga de datos severa (codificar el ID con el objetivo que intentamos predecir)

**Alternativas Avanzadas (y por qué fallan aquí):**
En escenarios de alta cardinalidad (p. ej., códigos ZIP), se suelen usar técnicas avanzadas. Sin embargo, requieren múltiples ejemplos por categoría para funcionar:
- **Codificación Leave-One-Out (LOO)**: Calcula la media del objetivo de *otros* registros con el mismo ID. Como la mayoría de los IDs aparecen solo una vez, LOO devolvería la media global (sin señal) o, para los pocos IDs repetidos, filtraría perfectamente el precio de la *única* otra venta, llevando a sobreajuste masivo.
- **Codificación Hash / CatBoost**: Estas también dependen de la recurrencia. Con `id` siendo casi único por fila, estas técnicas no pueden extraer un patrón estadístico.

### 5.3 Razón Práctica: El Modelo Memorizaría
Si incluyéramos `id` como característica, crearía un "atajo" peligroso para el modelo:
- **Memorización vs. aprendizaje**: El modelo puede aprender a identificar casas específicas (p. ej., "si id == 123456...") en lugar de aprender reglas generales (p. ej., "si sqft > 2000...")
- **El Problema del "Nuevo ID"**: Mientras el modelo *podría* seguir usando otras características, las reglas basadas en ID se vuelven inútiles o dañinas para cualquier propiedad no vista durante el entrenamiento
- **Resultado**: Obtenemos un modelo que rinde bien en los datos de entrenamiento (memorizando IDs) pero falla en datos nuevos (donde esos IDs no existen)

In [ ]:
# Demonstrate why ID can't be encodedn_unique_ids = df["id"].nunique()n_features = df.shape[1] - 2  # excluding id and priceprint(f"Current number of features: {n_features}")print(f"Number of unique IDs: {n_unique_ids:,}")print(f"\nIf we one-hot encoded 'id':")print(f"  New feature count: {n_features + n_unique_ids - 1:,}")print(f"  Features would outnumber samples!")print(f"\nThis is clearly not a viable approach.")

Current number of features: 19
Number of unique IDs: 21,436

If we one-hot encoded 'id':
  New feature count: 21,454
  Features would outnumber samples!

This is clearly not a viable approach.


## 6. Implicaciones para el Entrenamiento del Modelo
Ahora que comprendemos los datos, consideremos las implicaciones para el entrenamiento del modelo.

### 6.1 El No-Problema: "Fuga" por ID de Propiedad
Algunos podrían preocuparse: "¿Si la misma propiedad aparece en los conjuntos de entrenamiento y prueba, no es eso fuga de datos?"

**Respuesta: No, siempre que respetemos el tiempo.**

Si una propiedad se vendió en 2014 (entrenamiento) y nuevamente en 2015 (prueba), esto **no es fuga**. En un escenario de producción real, al predecir el precio de 2015, *sí tendríamos* acceso al historial de ventas de 2014. Esto imita la información real disponible en el momento de inferencia.

**Nota sobre Identificación:**
Aunque eliminamos la columna `id` para evitar la memorización explícita, el modelo podría seguir "identificando" propiedades a través de coordenadas espaciales de alta precisión (`lat`, `long`). Esto se llama **sobreajuste espacial**, pero es distinto a la preocupación de fuga de datos al usar información futura para predecir el pasado.

In [ ]:
# Can properties be uniquely identified by their características alone?feature_cols_for_id = ["sqft_living", "sqft_lot", "bedrooms", "bathrooms",                        "floors", "waterfront", "view", "condition", "grade",                       "yr_built", "lat", "long"]# Count unique característica combinationsunique_combinations = df.drop_duplicates(subset=feature_cols_for_id)print(f"Total records: {len(df):,}")print(f"Unique property IDs: {df['id'].nunique():,}")print(f"Unique feature combinations: {len(unique_combinations):,}")print(f"\nDifference: {df['id'].nunique() - len(unique_combinations):,} properties share features with others")

Total records: 21,613
Unique property IDs: 21,436
Unique feature combinations: 21,420

Difference: 16 properties share features with others


### 6.2 El Problema Real: Fuga Temporal
La verdadera preocupación con este conjunto de datos es la **fuga temporal**, que afecta a TODOS los registros, no solo a los IDs repetidos.

**El problema**: Una división aleatoria entrenamiento/prueba mezcla ventas de diferentes períodos de tiempo. El modelo puede aprender de las ventas de mayo de 2015 para predecir los precios de enero de 2014 — usando información "futura" para predecir el "pasado".

**La solución**: Usar una **división temporal** — entrenar con ventas más antiguas, probar con ventas más recientes.

## 7. ¿Debemos Conservar Todos los Registros?
Dado que los IDs repetidos representan ventas válidas y distintas, ¿debemos conservar todos los registros?

### Argumentos A FAVOR de Conservar Todos los Registros
1. **Cada uno es una observación válida**: Diferentes ventas en diferentes momentos con diferentes precios
2. **Más datos de entrenamiento**: Más ejemplos para que el modelo aprenda
3. **Captura la dinámica del mercado**: Los cambios de precio para la misma propiedad reflejan tendencias del mercado

### Argumentos A FAVOR de la Deduplicación (y por qué son en su mayoría inválidos aquí)
*¿Por qué alguien podría sugerir conservar solo el registro más reciente?*

1.  **Colisión de Características (Mismo X, Diferente Y)**:
    * *El argumento*: Si las características físicas son estáticas, el modelo ve la misma casa con dos precios diferentes.
    * *El Defecto*: Esto solo es cierto si ignoramos el **Tiempo**. Como `date` es una característica crucial, la entrada $X$ (Casa + Fecha) es en realidad única. La diferencia de precio se explica por la diferencia de tiempo.

2.  **Primacía de lo Reciente**:
    * *El argumento*: La venta más reciente es el valor "verdadero" actual.
    * *El Defecto*: Esto aplica a la *tasación* (¿cuánto vale *ahora*?), pero no necesariamente al *entrenamiento de Machine Learning*. Los datos más antiguos ayudan al modelo a aprender cómo evolucionan los precios a lo largo del tiempo. A menos que el mercado haya sufrido un cambio estructural fundamental (p. ej., antes vs. después de una crisis inmobiliaria), los datos históricos son una señal valiosa, no ruido.

### Recomendación
**Conservar todos los registros.**

Para este conjunto de datos específico, la deduplicación es **subóptima** y no es una estrategia sólida.

1.  **El "Ruido" es Señal**: El hecho de que la *misma* casa se vendiera a precios diferentes en diferentes momentos obliga al modelo a usar la característica `date` para explicar la diferencia. Esto ayuda al modelo a aprender las tendencias del mercado (inflación/deflación).
2.  **Horizonte Temporal Corto**: Este conjunto de datos abarca solo 1 año. La venta "más antigua" no es historia obsoleta; es un punto de datos reciente que ayuda a establecer la base de precios.
3.  **Escenario de Producción Realista**: Si una propiedad se vendió en el período de entrenamiento Y en el período de prueba, esto es realista — realmente sabríamos sobre la venta anterior al predecir la posterior.

In [ ]:
# With temporal split: how many properties appear in both train and test?df["date_parsed"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")df_sorted = df.sort_values("date_parsed")split_idx = int(len(df_sorted) * 0.8)train_df = df_sorted.iloc[:split_idx]test_df = df_sorted.iloc[split_idx:]train_ids = set(train_df["id"])test_ids = set(test_df["id"])shared_ids = train_ids.intersection(test_ids)print(f"Temporal split (80/20):")print(f"  Train: {len(train_df):,} records, {len(train_ids):,} unique properties")print(f"  Test: {len(test_df):,} records, {len(test_ids):,} unique properties")print(f"\nProperties appearing in BOTH sets: {len(shared_ids)}")print(f"\nThis is REALISTIC: we know about past sales when predicting future ones.")

Temporal split (80/20):
  Train: 17,290 records, 17,203 unique properties
  Test: 4,323 records, 4,323 unique properties

Properties appearing in BOTH sets: 90

This is REALISTIC: we know about past sales when predicting future ones.


## 8. Resumen y Recomendaciones

### Lo Que Encontramos

| Observación | Explicación |
|-------------|-------------|
| ~0.85% de los registros tienen IDs repetidos | La misma propiedad vendida múltiples veces |
| Las características físicas no cambian | Esperado — el mismo edificio |
| El precio y la fecha difieren | Transacciones diferentes |

### Acciones Recomendadas

| Acción | Justificación |
|--------|------------|
| **Eliminar la columna `id`** | Es un identificador, no una característica; no puede codificarse de forma significativa |
| **Conservar todos los registros** | Cada uno es una transacción de venta válida |
| **Usar división temporal** | Previene la fuga temporal (el verdadero problema) |

### Lo Que NO Hacer

| Enfoque Incorrecto | Por Qué Está Mal |
|-------------------|----------------|
| Codificar `id` con one-hot | Crea ~21,000 características; lleva a memorización |
| Eliminar registros "duplicados" | Pierde datos de entrenamiento válidos |
| Usar `GroupShuffleSplit` por `id` | No resuelve la fuga temporal; añade complejidad innecesaria |
| Usar división aleatoria entrenamiento/prueba | Causa fuga temporal |

In [ ]:
print("""FINAL PREPROCESSING GUIDANCE============================1. DROP the 'id' column:   df = df.drop(columns=['id'])2. KEEP all records (no deduplication needed)3. Use TEMPORAL train/test split:   df_sorted = df.sort_values('date')   split_idx = int(len(df_sorted) * 0.8)   train = df_sorted.iloc[:split_idx]   test = df_sorted.iloc[split_idx:]See notebook p3-03-temporal_leakage.ipynb for detailed explanation.""")


FINAL PREPROCESSING GUIDANCE

1. DROP the 'id' column:
   df = df.drop(columns=['id'])

2. KEEP all records (no deduplication needed)

3. Use TEMPORAL train/test split:
   df_sorted = df.sort_values('date')
   split_idx = int(len(df_sorted) * 0.8)
   train = df_sorted.iloc[:split_idx]
   test = df_sorted.iloc[split_idx:]

See notebook p3-03-temporal_leakage.ipynb for detailed explanation.



## 9. Matiz Avanzado: El ID como Clave para Ingeniería de Características
Debe hacerse una distinción crítica con respecto a la columna `id`. Si bien establecimos que `id` no debe usarse como característica en bruto (variable categórica) para prevenir la memorización, juega un papel diferente y vital en sistemas de producción del mundo real: **como Clave Relacional**.

### La Intuición
En un sistema de Valoración Automatizada (AVM) en producción, el historial de un activo específico suele ser el predictor más fuerte de su valor actual. Si una casa se vendió por \$500,000 el año pasado, eso es un poderoso "ancla" para el precio de hoy.

### La Implementación: Características de Rezago
En un escenario con historial profundo, no eliminaríamos `id`. En su lugar, lo usaríamos para crear **Características de Rezago (Lag Features)**.

1. Ordenar datos por `id` y `date`.
2. Desplazar la columna `price` para crear una característica `previous_price`.
3. Calcular el delta (`price` - `previous_price`).
4. **Luego** eliminar la columna `id` y entrenar sobre el *cambio* en el precio.

### El Desafío: Escasez de Señal vs. Valores Faltantes
Podrías preguntarte: *"¿No pueden los modelos modernos (como XGBoost o LightGBM) manejar los valores faltantes automáticamente?"*

**Sí, pueden.** Sin embargo, el problema aquí no es solo técnico (NaN causando errores) sino estadístico (Relación Señal-Ruido). Cuantifiquemos cuánta "señal" existe realmente en este conjunto de datos.

In [ ]:
# Let's attempt to engineer the "Previous Price" característica # to see if it provides enough signal for the modelo.# 1. Sort by ID and Date to ensure correct orderdf_lag = df.sort_values(by=['id', 'date']).copy()# 2. Create the lag característica: What was the price of this specific ID in the previous fila?df_lag['previous_price'] = df_lag.groupby('id')['price'].shift(1)# 3. Calculate metricstotal_rows = len(df_lag)rows_with_history = df_lag['previous_price'].notna().sum()missing_history = df_lag['previous_price'].isna().sum()print(f"Attempting to use ID for Lag Features:")print(f"---------------------------------------")print(f"Total records:               {total_rows:,}")print(f"Records with history (Signal): {rows_with_history:,}  ({(rows_with_history/total_rows)*100:.2f}%)")print(f"Records with NaN (Noise):    {missing_history:,}  ({(missing_history/total_rows)*100:.2f}%)")

Attempting to use ID for Lag Features:
---------------------------------------
Total records:               21,613
Records with history (Signal): 177  (0.82%)
Records with NaN (Noise):    21,436  (99.18%)


### Conclusión sobre Características Históricas
Como muestra el cálculo anterior, el **99.18%** de nuestras filas tendrían un `NaN` (valor faltante) para `previous_price`.

**El Veredicto:**
1.  **Manejo de NaN:** Si bien los modelos modernos *pueden* manejar NaN (p. ej., enviándolos por una rama predeterminada en un árbol), aún necesitan un número suficiente de ejemplos *no-NaN* para aprender el patrón subyacente.
2.  **Riesgo de Escasez:** Con solo ~176 ejemplos de "cómo el historial predice el precio futuro", el modelo probablemente no aprenderá una regla generalizable. Corre el riesgo de tratar la característica como ruido o sobreajustarse a esas 176 instancias específicas.
3.  **Realidad de Producción:** En una base de datos de empresa real con 10 años de ventas, este número podría ser del 50% o 60%. En ese caso, **definitivamente conservarías esta característica.**

**Recomendación:** Para este conjunto de datos educativo específico, eliminamos `id` para enfocarnos en las características físicas. En un proyecto real con historial más denso, usar `id` para construir características de rezago es una técnica estándar y poderosa.